# TP2 (a completer) : Arbre de decision — *Breast Cancer*

Remplacez chaque `...` et chaque `# TODO`. Corrige :
`../notebooks/TP2_arbre_decision.ipynb`.

**Objectif.** Predire si une tumeur est **maligne** ou **benigne** avec un arbre
**interpretable**, et evaluer en privilegiant le **rappel sur les cas malins**.

In [ ]:
# Cellule fournie : a executer telle quelle.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NAVY, ACCENT, GRAY = "#16284D", "#0EA5E9", "#5B6679"
RED = "#C0504D"
PALETTE = [ACCENT, NAVY, "#F79646", "#3FA45B", RED]
plt.rcParams.update({
    "figure.figsize": (7, 4.5), "font.size": 12,
    "axes.titlecolor": NAVY, "axes.titleweight": "bold",
    "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 120)
print("Environnement pret.")

## Etape 0 : charger les donnees (fournie)

In [ ]:
from sklearn.datasets import load_breast_cancer
ds = load_breast_cancer(as_frame=True)
X, y = ds.data, ds.target
CLASSES = list(ds.target_names)   # ['malignant', 'benign']  -> 0=malin, 1=benin
print(X.shape, CLASSES)

## 1. Exploration
**Consigne.** Affichez la repartition des classes (combien de malin / benin).

In [ ]:
# TODO : repartition de y (indice : value_counts + map vers CLASSES)
y.value_counts().rename(index=lambda i: CLASSES[i])

## 2. Modelisation
**Consigne.** Separez en train/test (25% test, `stratify=y`, `random_state=42`),
puis entrainez un `DecisionTreeClassifier` (`max_depth=3`, `min_samples_leaf=10`).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

arbre = arbre = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=42).fit(X_train, y_train)       # TODO : creer et entrainer l'arbre

## 3. Evaluation
**Consigne.** Affichez l'accuracy train et test, le `classification_report`
(avec `target_names=CLASSES`) et la **matrice de confusion**. Commentez le
**rappel** de la classe `malignant`.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# TODO : accuracy train / test
print("Accuracy train :", arbre.score(X_train, y_train))
print("Accuracy test  :", arbre.score(X_test, y_test))

y_pred = arbre.predict(X_test)
# TODO : afficher classification_report
print(classification_report(y_test, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4.2))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
plt.show()

## 4. Visualisation
**Consigne.** Tracez l'arbre (`plot_tree`) puis l'**importance des variables**
(barres horizontales des `feature_importances_` non nulles).

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(13, 7))
# TODO : plot_tree(arbre, feature_names=..., class_names=CLASSES, filled=True, ...)
fig, ax = plt.subplots(figsize=(13, 7))
plot_tree(arbre, feature_names=X.columns, class_names=CLASSES, filled=True, rounded=True, ax=ax)

plt.show()

In [ ]:
imp = pd.Series(arbre.feature_importances_, index=X.columns)
imp = imp[imp > 0].sort_values()
# TODO : barh de imp
imp.plot.barh(color=ACCENT)
plt.title("Importance des variables")
plt.show()

## 5. Prise de decision
**Consigne.** Sur 5 cas du test, affichez diagnostic reel, predit et
`predict_proba` de la classe benigne.

In [ ]:
ech = X_test.head(5)
# TODO : pred et proba, puis DataFrame comparatif
pred = arbre.predict(ech)
proba = arbre.predict_proba(ech)[:, 1]   

comparatif = pd.DataFrame({
    "reel": [CLASSES[i] for i in y_test.head(5)],
    "predit": [CLASSES[i] for i in pred],
    "proba_benin": proba.round(3),
})
comparatif

## A rendre
- Accuracy test + lecture de la matrice de confusion.

0.94. La matrice de confusion montre que la majorité des cas sont bien classés, avec très peu d'erreurs.

- Le rappel sur `malignant` et pourquoi il est prioritaire ici.

 c'est la métrique prioritaire car un faux négatif (tumeur maligne classée bénigne) signifie un cancer non détecté — bien plus grave qu'une fausse alerte. On vise donc un rappel maximal sur cette classe, quitte à accepter quelques fausses alertes.

- Les 2-3 variables les plus determinantes.

 worst perimeter, worst concave points, mean concave points.

**Bonus.** Faites varier `max_depth` (2, 3, 6, None) : ou commence le surapprentissage ?